# Simple Transformer Pipeline with Hugging Face

This notebook covers:

- What Hugging Face is
- Main Transformer model types
- Attention
- Self-attention
- Query, Key, Value
- Attention scores and softmax
- Context vectors
- Multi-head attention
- Masked / causal attention
- Cross-attention
- Pretrained models
- Tokenizers
- Built-in datasets
- BERT
- GPT-style models
- Llama and Mistral model family
- T5 encoder-decoder models
- A complete sentiment-analysis pipeline

The notebook is designed to run from top to bottom in Google Colab.

## 1. Install the required libraries

We use three main libraries:

- **transformers** → pretrained Transformer models and tokenizers
- **datasets** → ready-made datasets
- **torch** → tensor operations and neural-network utilities

In [ ]:
!pip install -q transformers datasets

## 2. Import libraries

In [ ]:
import math
import torch
import torch.nn as nn

from datasets import load_dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

print("PyTorch version:", torch.__version__)

## 3. What is Hugging Face?

Hugging Face is an AI ecosystem that provides pretrained models, tokenizers, datasets, and libraries.

A basic workflow looks like this:

```text
Text
  ↓
Tokenizer
  ↓
Token IDs
  ↓
Transformer Model
  ↓
Output
```

Examples of Transformer model families include **BERT, GPT, Llama, Mistral, T5, and BART**.

## 4. Main types of Transformer models

### Encoder-only
Examples: **BERT, DistilBERT, RoBERTa**

Main use: understanding text.

Typical tasks:
- Sentiment analysis
- Text classification
- Named entity recognition
- Extractive question answering

### Decoder-only
Examples: **GPT, GPT-2, Llama, Mistral**

Main use: generating text.

Typical tasks:
- Text generation
- Chat
- Code generation
- Next-token prediction

### Encoder-decoder
Examples: **T5, FLAN-T5, BART**

Main use: converting one text sequence into another.

Typical tasks:
- Translation
- Summarization
- Text-to-text generation

A simple memory rule:

```text
BERT → Understand
GPT / Llama / Mistral → Generate
T5 / BART → Transform
```

# Part 1 — Attention

Attention answers a simple question:

> Which words are important when understanding another word?

Example:

```text
The animal did not cross the road because it was tired.
```

When understanding **it**, the model should give more importance to **animal** than to unrelated words.

Transformer attention uses three vectors:

- **Query (Q)** → What am I looking for?
- **Key (K)** → What information do I contain?
- **Value (V)** → What information should I provide?

## 5. Create simple word embeddings

Neural networks work with numbers, not raw words.

Here we represent three words with random vectors. These are only for understanding the calculations.

In [ ]:
words = ["I", "love", "AI"]
print(words)

torch.manual_seed(42)
X = torch.randn(3, 4)

print("Word embeddings:")
print(X)
print("Shape:", X.shape)

The shape is:

```text
3 words × 4 numbers per word
```

So the tensor shape is **(3, 4)**.

## 6. Create Query, Key, and Value

Each input embedding is transformed into three different vectors:

```text
Input Embedding
   ├── Query
   ├── Key
   └── Value
```

In a real Transformer, the weight matrices are learned during training. Here we create random matrices only to see the process.

In [ ]:
embedding_size = 4

W_Q = torch.randn(embedding_size, embedding_size)
W_K = torch.randn(embedding_size, embedding_size)
W_V = torch.randn(embedding_size, embedding_size)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Query:")
print(Q)
print("
Key:")
print(K)
print("
Value:")
print(V)

## 7. Calculate attention scores

The first important calculation is:

```text
Q × Kᵀ
```

Every Query is compared with every Key.

With 3 words, we get a **3 × 3** score matrix.

In [ ]:
scores = Q @ K.T

print("Attention scores:")
print(scores)
print("Shape:", scores.shape)

## 8. Scale the scores

Scaled dot-product attention uses:

\[
	ext{Attention}(Q,K,V)=	ext{softmax}\left(rac{QK^T}{\sqrt{d}}ight)V
\]

The division by `sqrt(d)` helps keep the dot-product values under control.

In [ ]:
d = Q.shape[-1]
scaled_scores = scores / math.sqrt(d)

print("Scaled scores:")
print(scaled_scores)

## 9. Convert scores into attention weights with softmax

Softmax converts raw scores into values between 0 and 1.

Each row adds up to approximately 1.

In [ ]:
attention_weights = torch.softmax(scaled_scores, dim=-1)

print("Attention weights:")
print(attention_weights)
print("
Row sums:")
print(attention_weights.sum(dim=-1))

## 10. Create context vectors

Attention weights decide **where to focus**.

Values contain the **information to collect**.

The weighted combination is:

```text
Attention Weights × Values
```

This produces a contextual representation for every token.

In [ ]:
context = attention_weights @ V

print("Context vectors:")
print(context)
print("Shape:", context.shape)

## 11. Complete self-attention calculation

In **self-attention**, Query, Key, and Value all come from the same input sequence.

In [ ]:
X = torch.randn(3, 4)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = Q @ K.T
scores = scores / math.sqrt(Q.shape[-1])
attention_weights = torch.softmax(scores, dim=-1)
context = attention_weights @ V

print(context)

# Part 2 — Multi-Head Attention

One attention head gives one set of attention relationships.

Multi-head attention runs several attention calculations in parallel. Different heads can learn different patterns.

The outputs from the heads are combined into one representation.

## 12. Multi-head attention with PyTorch

In [ ]:
multi_head_attention = nn.MultiheadAttention(
    embed_dim=8,
    num_heads=2,
    batch_first=True
)

x = torch.randn(1, 4, 8)

output, weights = multi_head_attention(
    x,
    x,
    x,
    need_weights=True,
    average_attn_weights=False
)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention shape:", weights.shape)

For the attention tensor:

```text
[batch, heads, query_tokens, key_tokens]
```

For this example:

```text
[1, 2, 4, 4]
```

- 1 batch
- 2 attention heads
- 4 query positions
- 4 key positions

# Part 3 — Masked / Causal Attention

Decoder-only models such as GPT-style models generate text from left to right.

When predicting a token, the model must not look at future tokens.

Example:

```text
Token 1 → sees token 1
Token 2 → sees tokens 1, 2
Token 3 → sees tokens 1, 2, 3
Token 4 → sees tokens 1, 2, 3, 4
```

This is called **causal masking**.

## 13. Create a causal mask

In [ ]:
sequence_length = 4

causal_mask = torch.triu(
    torch.ones(sequence_length, sequence_length, dtype=torch.bool),
    diagonal=1
)

print(causal_mask)

## 14. Apply the causal mask

In [ ]:
masked_output, masked_weights = multi_head_attention(
    x,
    x,
    x,
    attn_mask=causal_mask,
    need_weights=True,
    average_attn_weights=False
)

print(masked_weights[0, 0])

# Part 4 — Cross-Attention

Cross-attention is commonly used in encoder-decoder Transformers.

The main difference is the source of Query, Key, and Value:

### Self-attention
```text
Q = same sequence
K = same sequence
V = same sequence
```

### Cross-attention
```text
Q = decoder
K = encoder
V = encoder
```

This lets the decoder look back at the encoder output while generating.

## 15. Cross-attention example

In [ ]:
encoder_output = torch.randn(1, 5, 8)
decoder_output = torch.randn(1, 3, 8)

cross_output, cross_weights = multi_head_attention(
    decoder_output,
    encoder_output,
    encoder_output,
    need_weights=True,
    average_attn_weights=False
)

print("Cross-attention output shape:", cross_output.shape)
print("Cross-attention weight shape:", cross_weights.shape)

# Part 5 — Pretrained Models

A pretrained model has already learned useful patterns from a large amount of data.

Instead of starting with random weights, we download learned weights using:

```python
from_pretrained(...)
```

A common workflow is:

```text
Pretraining → General language knowledge
Fine-tuning → Specific task
```

For example, a DistilBERT model can be fine-tuned for sentiment classification.

# Part 6 — Load a Ready-Made Dataset

We will use **SST-2**, a sentiment-analysis dataset.

The labels are:

```text
0 = Negative
1 = Positive
```

## 16. Load SST-2

In [ ]:
dataset = load_dataset(
    "stanfordnlp/sst2",
    split="validation"
)

print(dataset)

## 17. View an example

In [ ]:
print("First example:")
print(dataset[0])

print("
Sentence:")
print(dataset[0]["sentence"])

print("
Label:")
print(dataset[0]["label"])

# Part 7 — Simplest Pretrained Transformer Pipeline

We will use a DistilBERT checkpoint that has already been fine-tuned for sentiment analysis.

The complete flow is:

```text
Text
  ↓
Tokenizer
  ↓
DistilBERT
  ↓
Classification Head
  ↓
Positive / Negative
```

## 18. Create a sentiment-analysis pipeline

In [ ]:
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    "sentiment-analysis",
    model=model_name
)

text = "I absolutely loved this movie."
result = classifier(text)

print(result)

## 19. Try several sentences

In [ ]:
sentences = [
    "I love this course.",
    "This movie was terrible.",
    "The product is amazing.",
    "I hated the experience."
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print("Sentence:", sentence)
    print("Prediction:", result)
    print("-" * 50)

# Part 8 — What the Pipeline Does Internally

The `pipeline()` function hides several steps:

```text
Text
  ↓
Tokenizer
  ↓
Token IDs
  ↓
Transformer
  ↓
Logits
  ↓
Softmax
  ↓
Label
```

Now we perform those steps manually.

## 20. Load the tokenizer

A tokenizer converts text into token IDs that the model can process.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "I love this movie."
inputs = tokenizer(text, return_tensors="pt")

print(inputs)

## 21. View token IDs

In [ ]:
print(inputs["input_ids"])

## 22. Convert token IDs back into tokens

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

## 23. Attention mask

The tokenizer can also create an `attention_mask`.

Usually:

```text
1 = real token
0 = padding token
```

This padding mask is different from the causal mask used for future-token blocking in decoder-only models.

In [ ]:
print(inputs["attention_mask"])

## 24. Load the pretrained classification model

`AutoModelForSequenceClassification` means:

- **Auto** → automatically detect the correct architecture
- **Model** → load the Transformer
- **ForSequenceClassification** → attach/use a classification head

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)
print("Model loaded")

## 25. Run a forward pass

A forward pass sends the tokenized input through the model.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

print(outputs)

## 26. View logits

Logits are raw model scores. They are not probabilities yet.

In [ ]:
logits = outputs.logits
print(logits)

## 27. Convert logits into probabilities

Softmax converts the raw scores into probabilities.

In [ ]:
probabilities = torch.softmax(logits, dim=-1)
print(probabilities)

## 28. Get the final label

In [ ]:
prediction_id = probabilities.argmax(dim=-1).item()
label = model.config.id2label[prediction_id]

print("Prediction ID:", prediction_id)
print("Final prediction:", label)

# Part 9 — BERT Example

BERT is an **encoder-only** Transformer.

A classic BERT task is masked language modeling:

```text
Paris is the [MASK] of France.
```

BERT uses both left and right context to predict the missing token.

## 29. Fill a masked token with BERT

In [ ]:
fill_mask = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased"
)

results = fill_mask("Paris is the [MASK] of France.")

for result in results[:5]:
    print(result["token_str"], result["score"])

# Part 10 — GPT-2 Example

GPT-2 is a **decoder-only** Transformer.

It generates one token at a time using causal attention:

```text
Prompt → Next token → Next token → Next token → ...
```

## 30. Generate text with GPT-2

In [ ]:
gpt_name = "openai-community/gpt2"

gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_name)
gpt_model = AutoModelForCausalLM.from_pretrained(gpt_name)

prompt = "Artificial intelligence is"
inputs = gpt_tokenizer(prompt, return_tensors="pt")

output = gpt_model.generate(
    **inputs,
    max_new_tokens=20,
    do_sample=False,
    pad_token_id=gpt_tokenizer.eos_token_id
)

generated_text = gpt_tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(generated_text)

# Part 11 — Llama and Mistral

Llama and Mistral also belong to the **decoder-only / causal language model** family.

Conceptually, they are loaded with the same Hugging Face model class used for GPT-style generation:

```python
AutoTokenizer.from_pretrained(MODEL_NAME)
AutoModelForCausalLM.from_pretrained(MODEL_NAME)
```

The main difference is the checkpoint and the size of the model.

# Part 12 — T5 Encoder-Decoder Example

T5 contains both an encoder and a decoder.

```text
Input Text
   ↓
Encoder
   ↓
Encoder Hidden States
   ↓
Cross-Attention
   ↓
Decoder
   ↓
Output Text
```

We will use a small FLAN-T5 checkpoint.

## 31. Translate text with FLAN-T5

In [ ]:
t5_name = "google/flan-t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name)

text = "Translate English to German: I love machine learning."
inputs = t5_tokenizer(text, return_tensors="pt")

output = t5_model.generate(
    **inputs,
    max_new_tokens=30
)

result = t5_tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(result)

# Final Summary

## Attention
Decides which tokens are important for understanding another token.

## Self-Attention
Query, Key, and Value come from the same sequence.

## Query
What am I looking for?

## Key
What information do I contain?

## Value
What information should I provide?

## Context Vector
Weighted combination of Value vectors.

## Multi-Head Attention
Multiple attention calculations run in parallel.

## Masked / Causal Attention
Prevents decoder-only models from looking at future tokens.

Examples: GPT, Llama, Mistral.

## Encoder
Builds contextual representations of the input.

Examples: BERT, DistilBERT, RoBERTa.

## Decoder
Generates output token by token.

Examples: GPT, Llama, Mistral.

## Encoder-Decoder
Uses an encoder to process the input and a decoder to generate the output.

Examples: T5, FLAN-T5, BART.

## Cross-Attention
The decoder Query attends to encoder Keys and Values.

## Hugging Face
Provides pretrained models, tokenizers, datasets, pipelines, and other tools.

## Pretrained Model
A model whose weights have already been learned from data.

# Most Important Hugging Face Pattern

This is the core pattern to remember:

```text
Choose Model
    ↓
Load Tokenizer
    ↓
Load Model
    ↓
Tokenize Text
    ↓
Run Model
    ↓
Read Output
```

## 32. Complete classification pattern

In [ ]:
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "I love learning Transformers."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)
prediction = probabilities.argmax(dim=-1).item()
label = model.config.id2label[prediction]

print("Sentence:", text)
print("Prediction:", label)
print("Probabilities:", probabilities)

# Complete Flow

```text
Sentence
   ↓
Tokenizer
   ↓
Token IDs
   ↓
Embeddings
   ↓
Query, Key, Value
   ↓
Self-Attention
   ↓
Multi-Head Attention
   ↓
Contextual Representations
   ↓
Transformer Layers
   ↓
Task-Specific Output
```